In [83]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv("/content/drive/My Drive/AIML-Project/G6/weatherHistory.csv")
print("Shape:", df.shape)
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Shape: (96453, 12)


,Formatted Date,Summary,Precip Type,Temperature (C),Apparent Temperature (C),Humidity,Wind Speed (km/h),Wind Bearing (degrees),Visibility (km),Loud Cover,Pressure (millibars),Daily Summary
0,2006-04-01 00:00:00.000 +0200,Partly Cloudy,rain,9.472222,7.388889,0.89,14.1197,251.0,15.8263,0.0,1015.13,Partly cloudy throughout the day.
1,2006-04-01 01:00:00.000 +0200,Partly Cloudy,rain,9.355556,7.227778,0.86,14.2646,259.0,15.8263,0.0,1015.63,Partly cloudy throughout the day.
2,2006-04-01 02:00:00.000 +0200,Mostly Cloudy,rain,9.377778,9.377778,0.89,3.9284,204.0,14.9569,0.0,1015.94,Partly cloudy throughout the day.
3,2006-04-01 03:00:00.000 +0200,Partly Cloudy,rain,8.288889,5.944444,0.83,14.1036,269.0,15.8263,0.0,1016.41,Partly cloudy throughout the day.
4,2006-04-01 04:00:00.000 +0200,Mostly Cloudy,rain,8.755556,6.977778,0.83,11.0446,259.0,15.8263,0.0,1016.51,Partly cloudy throughout the day.


In [84]:
df.isnull().sum()

,0
Formatted Date,0
Summary,0
Precip Type,517
Temperature (C),0
Apparent Temperature (C),0
Humidity,0
Wind Speed (km/h),0
Wind Bearing (degrees),0
Visibility (km),0
Loud Cover,0


In [85]:
duplicates = df[df.duplicated()]
duplicates.shape

(24, 12)

In [86]:
df['Precip Type'] = df['Precip Type'].ffill() #ffill(), bfill(), mode() gives same values(assign to most rain)

In [87]:
df.isnull().sum()

,0
Formatted Date,0
Summary,0
Precip Type,0
Temperature (C),0
Apparent Temperature (C),0
Humidity,0
Wind Speed (km/h),0
Wind Bearing (degrees),0
Visibility (km),0
Loud Cover,0


In [88]:
df['Precip Type'].value_counts()

,count
Precip Type,
rain,85741
snow,10712


In [89]:
df['Summary'].value_counts()

,count
Summary,
Partly Cloudy,31733
Mostly Cloudy,28094
Overcast,16597
Clear,10890
Foggy,7148
Breezy and Overcast,528
Breezy and Mostly Cloudy,516
Breezy and Partly Cloudy,386
Dry and Partly Cloudy,86


In [90]:
df['Suitability'] = (df['Summary'] == 'Clear').astype(int)

In [91]:
df = df.drop_duplicates()

In [92]:
df = pd.get_dummies(
    df,
    columns=['Precip Type'],
    prefix='Precip',
    dtype=int
)

Handle Loud Cover

In [93]:
df['Loud Cover'].value_counts()

,count
Loud Cover,
0.0,96429


In [ ]:
# Extract time features BEFORE dropping the date (clear weather depends on season / time of day)
dt = pd.to_datetime(df['Formatted Date'], utc=True)
df['Month'] = dt.dt.month
df['Hour'] = dt.dt.hour
df = df.drop(columns=['Formatted Date', 'Summary', 'Daily Summary', 'Loud Cover'])

In [95]:
df.head()

,Temperature (C),Apparent Temperature (C),Humidity,Wind Speed (km/h),Wind Bearing (degrees),Visibility (km),Pressure (millibars),Suitability,Precip_rain,Precip_snow
0,9.472222,7.388889,0.89,14.1197,251.0,15.8263,1015.13,0,1,0
1,9.355556,7.227778,0.86,14.2646,259.0,15.8263,1015.63,0,1,0
2,9.377778,9.377778,0.89,3.9284,204.0,14.9569,1015.94,0,1,0
3,8.288889,5.944444,0.83,14.1036,269.0,15.8263,1016.41,0,1,0
4,8.755556,6.977778,0.83,11.0446,259.0,15.8263,1016.51,0,1,0


In [96]:
df = df.rename(columns={
    'Temperature (C)': 'Temperature',
    'Apparent Temperature (C)': 'Apparent_Temperature',
    'Wind Speed (km/h)': 'Wind_Speed',
    'Wind Bearing (degrees)': 'Wind_Bearing',
    'Visibility (km)': 'Visibility',
    'Pressure (millibars)': 'Pressure',
    'Precip_rain': 'Precip_Rain',
    'Precip_snow': 'Precip_Snow'
})

Scaling the Dataset

In [ ]:
# IMPORTANT: the old cell only called scaler.fit() and never transformed the data,
# so the SVM was trained on UNSCALED features (SVM is very sensitive to scale).
# Scaling is now done inside a Pipeline (after the train/test split) to avoid data leakage.

In [98]:
df.head()

,Temperature,Apparent_Temperature,Humidity,Wind_Speed,Wind_Bearing,Visibility,Pressure,Suitability,Precip_Rain,Precip_Snow
0,9.472222,7.388889,0.89,14.1197,251.0,15.8263,1015.13,0,1,0
1,9.355556,7.227778,0.86,14.2646,259.0,15.8263,1015.63,0,1,0
2,9.377778,9.377778,0.89,3.9284,204.0,14.9569,1015.94,0,1,0
3,8.288889,5.944444,0.83,14.1036,269.0,15.8263,1016.41,0,1,0
4,8.755556,6.977778,0.83,11.0446,259.0,15.8263,1016.51,0,1,0


In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
                             precision_recall_curve)

In [100]:
X = df.drop(columns=['Suitability'])
y = df['Suitability']

pd_X = pd.DataFrame(X)
pd_y = pd.Series(y)

pd_X.head()

,Temperature,Apparent_Temperature,Humidity,Wind_Speed,Wind_Bearing,Visibility,Pressure,Precip_Rain,Precip_Snow
0,9.472222,7.388889,0.89,14.1197,251.0,15.8263,1015.13,1,0
1,9.355556,7.227778,0.86,14.2646,259.0,15.8263,1015.63,1,0
2,9.377778,9.377778,0.89,3.9284,204.0,14.9569,1015.94,1,0
3,8.288889,5.944444,0.83,14.1036,269.0,15.8263,1016.41,1,0
4,8.755556,6.977778,0.83,11.0446,259.0,15.8263,1016.51,1,0


In [101]:
pd_y.head()

,Suitability
0,0
1,0
2,0
3,0
4,0


In [ ]:
# stratify=y keeps the Clear / Not-Clear ratio the same in every split (the data is imbalanced)
X_train, X_test, y_train, y_test = train_test_split(
    pd_X, pd_y, test_size=0.2, random_state=42, stratify=pd_y)

# small validation set (taken from train) used ONLY to pick the decision threshold
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)

print(f"Train: {len(X_tr)}, Validation: {len(X_val)}, Test: {len(X_test)}")
print("Class balance (Clear = 1):")
print(pd_y.value_counts(normalize=True).round(3))

In [ ]:
LABELS = ['Not Clear', 'Clear']

def plot_cm(y_true, y_pred, title):
    """Confusion matrix as counts + row-normalised percentages side by side."""
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    for ax, norm, fmt, sub in zip(axes, [None, 'true'], ['d', '.1%'],
                                  ['Counts', 'Normalised (per actual class)']):
        cm = confusion_matrix(y_true, y_pred, normalize=norm)
        ConfusionMatrixDisplay(cm, display_labels=LABELS).plot(
            ax=ax, cmap='Blues', values_format=fmt, colorbar=False)
        ax.set_title(sub)
        ax.set_xlabel('Predicted label')
        ax.set_ylabel('Actual label')
    fig.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

def report(y_true, y_pred, scores, title):
    print(f"\n{title}")
    print(f"  Accuracy : {accuracy_score(y_true, y_pred):.3f}")
    print(f"  Precision: {precision_score(y_true, y_pred):.3f}")
    print(f"  Recall   : {recall_score(y_true, y_pred):.3f}")
    print(f"  F1 Score : {f1_score(y_true, y_pred):.3f}")
    print(f"  ROC AUC  : {roc_auc_score(y_true, scores):.3f}")

In [ ]:
# Hyper-parameter search on a stratified SUBSAMPLE (RBF-SVM is slow on the full data),
# optimising F1 directly and using class_weight='balanced' for the imbalanced classes.
X_sub, _, y_sub, _ = train_test_split(
    X_tr, y_tr, train_size=12000, stratify=y_tr, random_state=42)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(kernel='rbf', class_weight='balanced', cache_size=1000))
])

param_grid = {
    'svc__C': [1, 10, 100],
    'svc__gamma': ['scale', 0.05, 0.2]
}

grid = GridSearchCV(pipe, param_grid, scoring='f1', cv=3, n_jobs=-1, verbose=1)
grid.fit(X_sub, y_sub)

print("Best params:", grid.best_params_)
print(f"Best CV F1 : {grid.best_score_:.3f}")

In [ ]:
# PCA must be applied to SCALED data
scaler_pca = StandardScaler().fit(X_train)
pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(scaler_pca.transform(X_train))
X_test_pca = pca.transform(scaler_pca.transform(X_test))

In [ ]:
rng = np.random.RandomState(42)
idx = rng.choice(len(X_train_pca), size=5000, replace=False)
Xp, yp = X_train_pca[idx], y_train.values[idx]

for kernel in ['linear', 'rbf']:

  model_svm_pca = SVC(kernel=kernel, class_weight='balanced').fit(Xp, yp)

  x_min, x_max = X_test_pca[:,0].min() - 1, X_test_pca[:, 0].max() + 1
  y_min, y_max = X_test_pca[:, 1].min() - 1, X_test_pca[:, 1].max() + 1
  xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1), np.arange(y_min, y_max, 0.1))
  Z = model_svm_pca.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

  plt.figure(figsize=(6, 4))
  plt.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
  plt.scatter(X_test_pca[::10, 0], X_test_pca[::10, 1], c=y_test.values[::10], cmap='coolwarm', edgecolors='k')
  plt.title(f"SVM Decision Boundary ({kernel}, PCA)")
  plt.xlabel("PCA Component 1")
  plt.ylabel("PCA Component 2")
  plt.show()

In [ ]:
# Final model with the best hyper-parameters (trained on X_tr; scaling happens inside the pipeline)
best_C = grid.best_params_['svc__C']
best_gamma = grid.best_params_['svc__gamma']

final_model = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(kernel='rbf', C=best_C, gamma=best_gamma,
                class_weight='balanced', cache_size=1000))
]).fit(X_tr, y_tr)

# Pick the threshold that maximises F1 on the VALIDATION set (never on the test set)
val_scores = final_model.decision_function(X_val)
prec, rec, thr = precision_recall_curve(y_val, val_scores)
f1_curve = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
best_thr = thr[np.argmax(f1_curve)]
print(f"Best threshold on validation set: {best_thr:.3f}  (default is 0)")

# Evaluate on the untouched test set
test_scores = final_model.decision_function(X_test)
pred_default = (test_scores > 0).astype(int)
pred_tuned = (test_scores >= best_thr).astype(int)

report(y_test, pred_default, test_scores, "SVM (RBF, balanced) - default threshold")
report(y_test, pred_tuned, test_scores, "SVM (RBF, balanced) - tuned threshold")

plot_cm(y_test, pred_default, "Confusion Matrix - default threshold")
plot_cm(y_test, pred_tuned, "Confusion Matrix - tuned threshold")